## ZERO SHOT INFERENCE

In [2]:
from groundingdino.util.inference import load_model, load_image, predict, annotate
from myfile import *
import os
import glob
import json
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch
import cv2

model_config_path = "C:/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_odvg.py"
model_checkpoint_path = "C:/GroundingDINO-Med-main/groundingdino_swint_ogc.pth"

data_dir = "C:/GroundingDINO-Med-main/floor_plans/sample/merged/test"
gt_json_path = os.path.join(data_dir, "_annotations.coco.json")

output_dir = "C:/GroundingDINO-Med-main/pred_images"
os.makedirs(output_dir, exist_ok=True)

pred_json_path = os.path.join(output_dir, "predictions.json")

device = "cuda" if torch.cuda.is_available() else "cpu"

caption = "door symbol. window symbol . zone"
box_threshold = 0.3
text_threshold = 0.25

cls_mapping = {
    "door symbol": 0,
    "window symbol": 1,
    "zone": 2
}

id2name = {v: k for k, v in cls_mapping.items()}

class_thresholds = {
    "door symbol": 0.2,
    "window symbol": 0.2,
    "zone": 0.2
}

model = load_model(model_config_path, model_checkpoint_path)
model.to(device)
model.eval()

cocoGt = COCO(gt_json_path)

predictions = []

for img_path in tqdm(glob.glob(os.path.join(data_dir, "*.jpg"))):

    file_name = os.path.basename(img_path)

    filename_to_id = {
        img["file_name"]: img["id"]
        for img in cocoGt.dataset["images"]
    }

    assert file_name in filename_to_id, f"{file_name} not found in GT JSON"
    image_id = filename_to_id[file_name]

    image_pil, image_tensor  = load_image(img_path)
    height, width = image_pil.shape[:2]
    
    boxes, logits, phrases = predict(
        model,
       image_tensor,
        caption,
        box_threshold,
        text_threshold
    )

    scores = [logit * 100 for logit in logits]
    draw_boxes = []
    draw_labels = []


    for box, phrase, score  in zip(boxes, phrases, scores):
        
        class_name = phrase.split("(")[0].strip()
        if class_name not in cls_mapping:
            continue
        
        threshold = class_thresholds.get(class_name, 0.3)

        if score < threshold:
            continue
   
        x1, y1, x2, y2 = cxcywh2xyxy(box, width, height)
        w, h = x2 - x1, y2 - y1

        predictions.append({
            "image_id": int(image_id),
            "category_id": int(cls_mapping[class_name]),
            "bbox": [float(x1), float(y1), float(w), float(h)],
            "area": float(w * h),
            "score": float(score)
        })

        draw_boxes.append(box)
        draw_labels.append(f"{class_name}({score:.2f})")

    if len(draw_boxes) > 0:
        draw_dict = {
            "boxes": draw_boxes,
            "labels": draw_labels,
            "size": [width, height]
        }
        
        annotated = annotate(image_pil, boxes, logits, phrases)
        cv2.imwrite(os.path.join(output_dir, file_name), annotated[:, :, ::-1])  

with open(pred_json_path, "w") as f:
    json.dump(predictions, f, indent=4)

print(f"Saved predictions to {pred_json_path}")


cocoDt = cocoGt.loadRes(predictions)
cocoEval = COCOeval(cocoGt, cocoDt, iouType="bbox")
#cocoEval.params.iouThrs = np.array([0.5])
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

precisions = cocoEval.eval['precision']  # shape: [T,R,K,A,M]
recalls = cocoEval.eval['recall']        # shape: [T,K,A,M]
# mean over all recall points, categories, area=all, maxDets=100
p = np.mean(precisions[0, :, :, 0, -1])
r = np.mean(recalls[0, :, 0, -1])

print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")

for cls_id, cls_name in id2name.items():

    cocoEval_cls = COCOeval(cocoGt, cocoDt, iouType="bbox")
    #cocoEval_cls.params.iouThrs = np.array([0.3])
    cocoEval_cls.params.catIds = [cls_id]

    print("\n==============================")
    print(f"Class: {cls_name}")

    cocoEval_cls.evaluate()
    cocoEval_cls.accumulate()
    cocoEval_cls.summarize()

    precisions = cocoEval_cls.eval['precision']  # shape: [T,R,K,A,M]
    recalls = cocoEval_cls.eval['recall']        # shape: [T,K,A,M]
    # mean over all recall points, categories, area=all, maxDets=100
    p = np.mean(precisions[0, :, :, 0, -1])
    r = np.mean(recalls[0, :, 0, -1])

    print(f"Precision: {p:.3f}")
    print(f"Recall:    {r:.3f}")


final text_encoder_type: bert-base-uncased
load tokenizer done.
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


100%|██████████| 40/40 [00:06<00:00,  6.13it/s]

Saved predictions to C:/GroundingDINO-Med-main/pred_images\predictions.json
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.003
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.006
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.003
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] 

## Validation and Test INFERENCE

In [1]:
from tools.inference_on_a_image import *
from myfile import *
import os
import glob
import json
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch


model_config_path = "C:/GroundingDINO-Med-main/OpenGroundingDino/tools/GroundingDINO_SwinT_OGC.py"
model_checkpoint_path = "C:/GroundingDINO-Med-main/floor_plans/output/unfreeze/sample25/checkpoint_best_regular.pth"

data_dir = "C:/GroundingDINO-Med-main/floor_plans/sample/merged/test"
gt_json_path = os.path.join(data_dir, "_annotations.coco.json")

output_dir = "C:/GroundingDINO-Med-main/pred_images"
os.makedirs(output_dir, exist_ok=True)

pred_json_path = os.path.join(output_dir, "predictions.json")


device = "cuda" if torch.cuda.is_available() else "cpu"

caption = "door symbol . window symbol . zone"
box_threshold = 0.3
text_threshold = 0.25

cls_mapping = {
    "door symbol": 0,
    "window symbol": 1,
    "zone": 2
}

id2name = {v: k for k, v in cls_mapping.items()}

class_thresholds = {
    "door symbol": 0.2,
    "window symbol": 0.2,
    "zone": 0.2
}

model = load_model(model_config_path, model_checkpoint_path)
model.to(device)
model.eval()

cocoGt = COCO(gt_json_path)

predictions = []

for img_path in tqdm(glob.glob(os.path.join(data_dir, "*.jpg"))):

    file_name = os.path.basename(img_path)
    img_cv2 = cv2.imread(img_path)
    filename_to_id = {
        img["file_name"]: img["id"]
        for img in cocoGt.dataset["images"]
    }

    assert file_name in filename_to_id, f"{file_name} not found in GT JSON"
    image_id = filename_to_id[file_name]

    image_pil, image_tensor, width, height = load_image(img_path)

    boxes, phrases = get_grounding_output(
        model,
        image_tensor,
        caption,
        box_threshold,
        text_threshold
    )

    draw_boxes = []
    draw_labels = []

    for box, phrase in zip(boxes, phrases):

        color = tuple(np.random.randint(0, 255, size=3).tolist())
        class_name = phrase.split("(")[0].strip()
        if class_name not in cls_mapping:
            continue

        score = float(phrase.split("(")[1].replace(")", ""))
        threshold = class_thresholds.get(class_name, 0.3)

        if score < threshold:
            continue

        x1, y1, x2, y2 = cxcywh2xyxy(box, width, height)
 
        w, h = x2 - x1, y2 - y1

        predictions.append({
            "image_id": int(image_id),
            "category_id": int(cls_mapping[class_name]),
            "bbox": [float(x1), float(y1), float(w), float(h)],
            "area": float(w * h),
            "score": float(score),
            "iscrowd": 0
        })

        draw_boxes.append(box)
        draw_labels.append(f"{class_name}({score:.2f})")
        
        draw_annotation(image_pil, (x1, y1, x2, y2), phrase)
    image_pil.save(os.path.join(output_dir, file_name))

with open(pred_json_path, "w") as f:
    json.dump(predictions, f, indent=4)

print(f"Saved predictions to {pred_json_path}")

cocoDt = cocoGt.loadRes(predictions)
cocoEval = COCOeval(cocoGt, cocoDt, iouType="bbox")
#cocoEval.params.iouThrs = np.array([0.5])
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

precisions = cocoEval.eval['precision']  # shape: [T,R,K,A,M]
recalls = cocoEval.eval['recall']        # shape: [T,K,A,M]
# mean over all recall points, categories, area=all, maxDets=100
p = np.mean(precisions[0, :, :, 0, -1])
r = np.mean(recalls[0, :, 0, -1])

print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")

for cls_id, cls_name in id2name.items():

    cocoEval_cls = COCOeval(cocoGt, cocoDt, iouType="bbox")
    #cocoEval_cls.params.iouThrs = np.array([0.3])
    cocoEval_cls.params.catIds = [cls_id]

    print("\n==============================")
    print(f"Class: {cls_name}")

    cocoEval_cls.evaluate()
    cocoEval_cls.accumulate()
    cocoEval_cls.summarize()

    precisions = cocoEval_cls.eval['precision']  # shape: [T,R,K,A,M]
    recalls = cocoEval_cls.eval['recall']        # shape: [T,K,A,M]
    # mean over all recall points, categories, area=all, maxDets=100
    p = np.mean(precisions[0, :, :, 0, -1])
    r = np.mean(recalls[0, :, 0, -1])

    print(f"Precision: {p:.3f}")
    print(f"Recall:    {r:.3f}")

c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4319.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased
load tokenizer done.
<All keys matched successfully>
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


  0%|          | 0/40 [00:00<?, ?it/s]c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\transformers\modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\limfo\.conda\envs\opendino1\lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
c:\GroundingDINO-Med-main\OpenGroundingDino\groundingdino\models\Ground

Saved predictions to C:/GroundingDINO-Med-main/pred_images\predictions.json
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.21s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.470
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.779
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.469
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.198
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.304
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.498
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.080
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.480
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] 